# 🛡️ Autonomy Calibration Environment — GRPO Training Notebook

**OpenEnv India Hackathon 2026**

This notebook fine-tunes a small LLM (`Qwen/Qwen2.5-0.5B-Instruct`) using **Group Relative Policy Optimization (GRPO)** via Hugging Face TRL.

The reward signal comes directly from the live Autonomy Calibration Environment API.

### What the agent learns:
- **Email Triage**: Classify phishing/spam/urgent emails and choose correct response actions
- **DevOps Incident**: Diagnose production incidents and apply safe recovery procedures
- **Financial Requests**: Detect fraud signals and escalate or approve transfers appropriately

### Architecture
```
LLM Policy → generates action → Environment API → reward signal → GRPO update
```

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q trl>=0.8.0 transformers>=4.40.0 accelerate peft bitsandbytes
!pip install -q requests matplotlib numpy huggingface_hub

# Optional: Unsloth for faster training on Colab GPU
# !pip install -q unsloth

print('✅ Dependencies installed')

## 2. Configuration

In [ ]:
import os
import json
import requests
import random
from typing import List

# ── Environment Config ────────────────────────────────────────────────
# Option A: Use your live Hugging Face Space
ENV_API = "https://joy0021-autonomy-calibration-benchmark.hf.space/api"

# Option B: Local server (if running inference.py locally)
# ENV_API = "http://localhost:8000/api"

TASKS = ["email_triage", "devops_incident", "financial_request"]

# ── Model Config ──────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # Small, fast, runs on free T4
MAX_NEW_TOKENS = 64
LEARNING_RATE = 5e-6
NUM_TRAIN_STEPS = 100
BATCH_SIZE = 4

print(f'✅ Config loaded. Environment: {ENV_API}')
print(f'   Model: {MODEL_NAME}')
print(f'   Tasks: {TASKS}')

## 3. Environment Interface

In [ ]:
def env_reset(task: str = None) -> dict:
    """Reset the environment and return initial observation."""
    payload = {"task": task} if task else {}
    r = requests.post(f"{ENV_API}/reset", json=payload, timeout=20)
    r.raise_for_status()
    return r.json()

def env_step(action_type: str) -> dict:
    """Send an action and get back (observation, reward, done, info)."""
    r = requests.post(f"{ENV_API}/step", json={"type": action_type}, timeout=20)
    r.raise_for_status()
    return r.json()

def run_episode_with_model(model, tokenizer, task: str, device: str) -> float:
    """Run one full episode using the LLM to select actions. Returns episode score."""
    obs = env_reset(task)
    total_reward = 0.0
    done = False
    max_steps = 6
    step = 0
    
    while not done and step < max_steps:
        available = obs.get("available_actions", [])
        if not available:
            break
        
        # Build prompt for LLM
        prompt = (
            f"You are an AI agent. Read the situation and choose one action.\n"
            f"Situation: {obs.get('prompt', '')}\n"
            f"Available actions: {available}\n"
            f"Respond with ONLY the action name (one of the available actions):"
        )
        
        # Get model prediction
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
        with __import__('torch').no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.3,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        
        # Extract action from generated text
        action = None
        for a in available:
            if a in generated:
                action = a
                break
        if action is None:
            action = random.choice(available)
        
        # Step the environment
        result = env_step(action)
        reward = result.get("reward", {}).get("value", 0.0)
        total_reward += reward
        obs = result.get("observation", {})
        done = result.get("done", False)
        step += 1
    
    return total_reward

# Quick sanity check
try:
    obs = env_reset("email_triage")
    print(f'✅ Environment connection OK!')
    print(f'   Task: {obs["task_id"]} | Step: {obs["step"]}')
    print(f'   Available actions: {obs["available_actions"]}')
except Exception as e:
    print(f'⚠️  Connection failed: {e}')
    print('   Make sure your HF Space is running or use local server.')

## 4. Load Model & Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Device: {device}')

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

# Apply LoRA for parameter-efficient fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ Model loaded with LoRA adapters')

## 5. Build GRPO Training Dataset

GRPO requires a dataset of prompts. We collect prompts by resetting the environment and recording the initial observations.

In [ ]:
from datasets import Dataset

def collect_prompts(n_per_task: int = 20) -> List[dict]:
    """Collect prompts from the environment to use as GRPO training seeds."""
    records = []
    for task in TASKS:
        for i in range(n_per_task):
            try:
                obs = env_reset(task)
                available = obs.get("available_actions", [])
                prompt = (
                    f"You are an AI agent. Read the situation and choose one action.\n"
                    f"Situation: {obs.get('prompt', '')}\n"
                    f"Available actions: {available}\n"
                    f"Respond with ONLY the action name (one of the available actions):"
                )
                records.append({
                    "prompt": prompt,
                    "task_id": task,
                    "available_actions": json.dumps(available)
                })
            except Exception as e:
                print(f'Warning: {e}')
    return records

print('Collecting training prompts from environment...')
records = collect_prompts(n_per_task=15)
dataset = Dataset.from_list(records)
print(f'✅ Dataset ready: {len(dataset)} prompts across 3 tasks')
print(dataset)

## 6. Define the Reward Function

This is the core of GRPO. The reward function calls the live environment, sends the model's generated action, and returns the environment's reward signal.

In [ ]:
import re

def autonomy_reward_fn(completions, prompts=None, **kwargs) -> List[float]:
    """
    GRPO Reward Function.
    Sends the model's generated action to the environment API and
    returns the clamped reward [0.01, 0.99].
    """
    rewards = []
    task_ids = kwargs.get("task_id", ["email_triage"] * len(completions))
    available_actions_list = kwargs.get("available_actions", ["[]"] * len(completions))
    
    for i, (completion, task_id) in enumerate(zip(completions, task_ids)):
        # Extract the text from completion dict if needed
        if isinstance(completion, list):
            text = completion[0].get("content", "") if completion else ""
        else:
            text = str(completion)
        
        available = json.loads(available_actions_list[i]) if isinstance(available_actions_list[i], str) else available_actions_list[i]
        
        # Extract the action from generated text
        action = None
        for a in available:
            if a in text:
                action = a
                break
        
        if action is None:
            # Invalid action: give minimum reward
            rewards.append(0.01)
            continue
        
        # Query the environment for reward
        try:
            env_reset(task_id)
            result = env_step(action)
            reward = result.get("reward", {}).get("value", 0.01)
            rewards.append(float(reward))
        except Exception:
            rewards.append(0.01)
    
    return rewards

print('✅ Reward function defined')
print('   Input: model completion text')
print('   Output: float reward in [0.01, 0.99] from live environment')

## 7. Configure & Run GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./autonomy_grpo_checkpoints",
    num_train_epochs=1,
    max_steps=NUM_TRAIN_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    max_new_tokens=MAX_NEW_TOKENS,
    max_prompt_length=512,
    num_generations=4,            # GRPO group size
    temperature=0.7,
    logging_steps=5,
    save_steps=50,
    gradient_accumulation_steps=2,
    report_to="none",             # Switch to "wandb" if using Weights & Biases
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=autonomy_reward_fn,
    args=grpo_config,
    train_dataset=dataset,
)

print('🚀 Starting GRPO training...')
print(f'   Steps: {NUM_TRAIN_STEPS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}')
train_result = trainer.train()
print('✅ Training complete!')
print(train_result)

## 8. Evaluate: Baseline vs Trained Agent

In [ ]:
print('Evaluating trained agent vs rule-based baseline...')

EVAL_EPISODES = 5
results = {}

for task in TASKS:
    trained_scores = []
    for ep in range(EVAL_EPISODES):
        score = run_episode_with_model(trainer.model, tokenizer, task, device)
        trained_scores.append(score)
        print(f'  {task} ep{ep}: {score:.3f}')
    results[task] = trained_scores

print('\n=== EVALUATION RESULTS ===')
for task, scores in results.items():
    avg = sum(scores) / len(scores)
    print(f'  {task:30s}: {avg:.4f}')

## 9. Plot Learning Curves

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Extract training log
log_history = trainer.state.log_history
steps, rewards, losses = [], [], []

for entry in log_history:
    if "reward" in entry:
        steps.append(entry.get("step", 0))
        rewards.append(entry["reward"])
    if "loss" in entry:
        losses.append(entry["loss"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GRPO Training — Autonomy Calibration Environment', fontsize=14, fontweight='bold')

# Reward curve
if rewards:
    axes[0].plot(steps, rewards, color='#27AE60', linewidth=2)
    axes[0].set_title('Episode Reward During Training')
    axes[0].set_xlabel('Training Step')
    axes[0].set_ylabel('Reward (0.01–0.99)')
    axes[0].grid(True, alpha=0.3)

# Loss curve
if losses:
    axes[1].plot(range(len(losses)), losses, color='#E74C3C', linewidth=2)
    axes[1].set_title('Policy Loss During Training')
    axes[1].set_xlabel('Training Step')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('grpo_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved grpo_training_curves.png')

## 10. Push Trained Model to Hugging Face Hub (Optional)

In [ ]:
from huggingface_hub import login

HF_TOKEN = "your_hf_token_here"  # Replace with your token
HF_REPO = "JOY0021/autonomy-calibration-agent"  # Your target repo

if HF_TOKEN != "your_hf_token_here":
    login(token=HF_TOKEN)
    trainer.model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f'✅ Model pushed to https://huggingface.co/{HF_REPO}')
else:
    trainer.save_model("./autonomy_grpo_final")
    print('✅ Model saved locally to ./autonomy_grpo_final')